In [12]:
import pandas as pd

In [13]:
df1 = pd.read_csv("Emaint Data.csv")
df2 = pd.read_csv("Coffee Downtime and Maintenance Data.csv")

In [14]:
df1.columns = df1.columns.str.strip().str.lower().str.replace(" ", "_")
df2.columns = df2.columns.str.strip().str.lower().str.replace(" ", "_")

In [15]:
def make_text_emaint(row):
    return f"""
    Asset ID: {row.get('asset_id', '')}
    Equipment Description: {row.get('equipment_description', '')}
    Line No: {row.get('line_no', '')}
    Work Order Type: {row.get('wo_type', '')}
    Failure Type: {row.get('failure_type', '')}
    Downtime: {row.get('downtime', '')}
    Date: {row.get('wo_date', '')}
    """.strip()

df1["combined_text"] = df1.apply(make_text_emaint, axis=1)

In [16]:
def make_text_coffee(row):
    return f"""
    Plant: {row.get('plantname', '')}
    Line: {row.get('linename', '')}
    Shift: {row.get('shiftname', '')}
    Order Number: {row.get('activeordernumber', '')}
    Shift Start Date: {row.get('shiftstartdate', '')}
    Material: {row.get('materialdescr', '')}
    Uptime: {row.get('uptime', '')}
    Total Downtime: {row.get('totaldowntime', '')}
    Unplanned Downtime: {row.get('unplanneddowntime', '')}
    Planned Downtime: {row.get('planneddowntime', '')}
    Other Downtime: {row.get('otherdowntime', '')}
    Changeover: {row.get('changeover', '')}
    Quantity In: {row.get('qtyin', '')}
    Quantity Out: {row.get('qtyout', '')}
    Quantity Processed: {row.get('qtyprocessed', '')}
    Quantity Rejected: {row.get('qtyrejected', '')}
    Audit Status: {row.get('auditstatus', '')}
    Data Source: {row.get('datasource', '')}
    """.strip()

df2["combined_text"] = df2.apply(make_text_coffee, axis=1)

In [17]:
# Prepare eMaint fields for trend analysis
df1["wo_date"] = pd.to_datetime(df1["wo_date"], errors="coerce")
df1["downtime"] = pd.to_numeric(df1["downtime"], errors="coerce")

# Keep only rows with valid dates for time trends
df1_trend = df1.dropna(subset=["wo_date"]).copy()

# Create month column
df1_trend["month"] = df1_trend["wo_date"].dt.to_period("M").astype(str)

In [18]:
failure_trend = (
    df1_trend.groupby(["month", "failure_type"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

print("FAILURE TYPE TREND BY MONTH")
print(failure_trend)

FAILURE TYPE TREND BY MONTH
failure_type  Breakdown  Process Failure  not found
month                                              
2025-01              28              391          0
2025-02              28              489          0
2025-03              52              508          0
2025-04              23              532          0
2025-05              31              521          1
2025-06              43              517          1
2025-07              38              560          1
2025-08              36              471          0
2025-09              31              443          0
2025-10              41              573          0
2025-11              35              471          0
2025-12              36              491          0
2026-01              32              446          0
2026-02               5              120          0


In [19]:
wo_trend = (
    df1_trend.groupby(["month", "wo_type"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

print("\nWORK ORDER TYPE TREND BY MONTH")
print(wo_trend)


WORK ORDER TYPE TREND BY MONTH
wo_type  Corrective Maintenance  \
month                             
2025-01                     132   
2025-02                     127   
2025-03                     126   
2025-04                     118   
2025-05                     142   
2025-06                     155   
2025-07                     151   
2025-08                     107   
2025-09                     130   
2025-10                     186   
2025-11                     141   
2025-12                     143   
2026-01                     133   
2026-02                      50   

wo_type  Critical Safety Hazard Corrective Maintenance  Food Safety Request  \
month                                                                         
2025-01                                              0                    1   
2025-02                                              0                    0   
2025-03                                              0                    0   
2025-04     

In [20]:
failure_summary = (
    df1_trend.groupby("failure_type")
    .agg(
        work_order_count=("wo_no.", "count"),
        total_downtime_hours=("downtime", "sum"),
        avg_downtime_hours=("downtime", "mean")
    )
    .sort_values(by="work_order_count", ascending=False)
)

print("\nFAILURE TYPE SUMMARY")
print(failure_summary)


FAILURE TYPE SUMMARY
                 work_order_count  total_downtime_hours  avg_downtime_hours
failure_type                                                               
Process Failure              6533               4050.87            0.620063
Breakdown                     459                740.71            1.613747
not found                       3                  1.50            0.500000


In [21]:
wo_summary = (
    df1_trend.groupby("wo_type")
    .agg(
        work_order_count=("wo_no.", "count"),
        total_downtime_hours=("downtime", "sum"),
        avg_downtime_hours=("downtime", "mean")
    )
    .sort_values(by="work_order_count", ascending=False)
)

print("\nWORK ORDER TYPE SUMMARY")
print(wo_summary)


WORK ORDER TYPE SUMMARY
                                               work_order_count  \
wo_type                                                           
Operator Adjustment                                        3364   
Corrective Maintenance                                     1841   
Rotary Valve                                                589   
Manual Product Move                                         511   
Minor Breakdown Maintenance                                 437   
Printer Issue                                               187   
Food Safety Request                                          27   
Major Breakdown Maintenance                                  22   
Temporary Repair                                             14   
Critical Safety Hazard Corrective Maintenance                 2   

                                               total_downtime_hours  \
wo_type                                                               
Operator Adjustment         

In [22]:
equipment_summary = (
    df1_trend.groupby(["asset_id", "equipment_description", "line_no"])
    .agg(
        work_order_count=("wo_no.", "count"),
        total_downtime_hours=("downtime", "sum"),
        avg_downtime_hours=("downtime", "mean")
    )
    .sort_values(by="work_order_count", ascending=False)
)

print("\nREPEATED ISSUES BY ASSET / EQUIPMENT / LINE")
print(equipment_summary.head(15))


REPEATED ISSUES BY ASSET / EQUIPMENT / LINE
                                                                          work_order_count  \
asset_id   equipment_description                          line_no                            
FW000852   70-FL-40 Fresco G14, Form Fill  LINE 40        40- Retail FLV               514   
FW000631   70-FL-24 Fresco Retail Packaging Line 24       24-Retail1                   324   
FW000362   70-FL-22 5LB. General #2 Bagger Filler Line 22 22- 5LB                      222   
FW000593   70-FL-21A Frac 1 Packaging Line 21 Side A      21-Frac 1                    190   
FW000339   70-FL-23 Urn General Bagger Filler             23- Urn 1                    161   
FW001270   70-FL-27 Rovema SBS250                         27-Retail                    161   
FW000278   30-RS-05 RT3000 Roaster JUPITER                RT3000                       139   
FW200071   Roaster Neptune # 8                            Neptune                      125   
FW000817   70-F

In [23]:
df1["source"] = "emaint"
df2["source"] = "coffee"

In [24]:
combined_df = pd.concat([
    df1[["source", "combined_text"]],
    df2[["source", "combined_text"]]
], ignore_index=True)

print(combined_df["source"].value_counts())

source
coffee    50431
emaint     6995
Name: count, dtype: int64


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

emaint_df = combined_df[combined_df["source"] == "emaint"].copy()
coffee_df = combined_df[combined_df["source"] == "coffee"].copy()

vectorizer_emaint = TfidfVectorizer(stop_words="english")
X_emaint = vectorizer_emaint.fit_transform(emaint_df["combined_text"].fillna(""))

vectorizer_coffee = TfidfVectorizer(stop_words="english")
X_coffee = vectorizer_coffee.fit_transform(coffee_df["combined_text"].fillna(""))

In [26]:
def retrieve_both_sources(query, top_k_each=5):
    query_vec_emaint = vectorizer_emaint.transform([query])
    similarities_emaint = cosine_similarity(query_vec_emaint, X_emaint).flatten()
    top_indices_emaint = similarities_emaint.argsort()[-top_k_each:][::-1]
    top_emaint = emaint_df.iloc[top_indices_emaint].assign(score=similarities_emaint[top_indices_emaint])

    query_vec_coffee = vectorizer_coffee.transform([query])
    similarities_coffee = cosine_similarity(query_vec_coffee, X_coffee).flatten()
    top_indices_coffee = similarities_coffee.argsort()[-top_k_each:][::-1]
    top_coffee = coffee_df.iloc[top_indices_coffee].assign(score=similarities_coffee[top_indices_coffee])

    return top_emaint, top_coffee

In [27]:
top_emaint, top_coffee = retrieve_both_sources(
    "What patterns are causing repeated downtime and what maintenance actions should be considered?",
    top_k_each=5
)

print("EMAINT RESULTS")
print(top_emaint[["source", "score", "combined_text"]])

print("\nCOFFEE RESULTS")
print(top_coffee[["source", "score", "combined_text"]])

EMAINT RESULTS
      source     score                                      combined_text
4726  emaint  0.301520  Asset ID: FW000512\n    Equipment Description:...
6709  emaint  0.298375  Asset ID: FW000512\n    Equipment Description:...
6376  emaint  0.297859  Asset ID: FW000512\n    Equipment Description:...
1405  emaint  0.209929  Asset ID: FW000512\n    Equipment Description:...
2764  emaint  0.197257  Asset ID: FW001013\n    Equipment Description:...

COFFEE RESULTS
       source     score                                      combined_text
37066  coffee  0.351888  Plant: Northlake\n    Line: L40\n    Shift: SH...
37384  coffee  0.348827  Plant: Northlake\n    Line: L22\n    Shift: SH...
37674  coffee  0.348827  Plant: Northlake\n    Line: L22\n    Shift: SH...
37542  coffee  0.348827  Plant: Northlake\n    Line: L22\n    Shift: SH...
37675  coffee  0.348340  Plant: Northlake\n    Line: L22\n    Shift: SH...


In [28]:
query = "What patterns are causing repeated downtime and what maintenance actions should be considered?"

print("QUESTION:")
print(query)

print("\nTOP EMAINT MATCHES:")
for i in range(len(top_emaint)):
    print(f"\n--- eMaint {i+1} ---")
    print(top_emaint.iloc[i]["combined_text"])

print("\nTOP COFFEE MATCHES:")
for i in range(len(top_coffee)):
    print(f"\n--- Coffee {i+1} ---")
    print(top_coffee.iloc[i]["combined_text"])

QUESTION:
What patterns are causing repeated downtime and what maintenance actions should be considered?

TOP EMAINT MATCHES:

--- eMaint 1 ---
Asset ID: FW000512
    Equipment Description: Maintenance Shop
    Line No: Maintenance Shop
    Work Order Type: Corrective Maintenance
    Failure Type: Process Failure
    Downtime: 1.0
    Date: 9/30/2025

--- eMaint 2 ---
Asset ID: FW000512
    Equipment Description: Maintenance Shop
    Line No: Maintenance Shop
    Work Order Type: Corrective Maintenance
    Failure Type: Process Failure
    Downtime: 0.5
    Date: 1/21/2026

--- eMaint 3 ---
Asset ID: FW000512
    Equipment Description: Maintenance Shop
    Line No: Maintenance Shop
    Work Order Type: Corrective Maintenance
    Failure Type: Process Failure
    Downtime: 1.0
    Date: 12/30/2025

--- eMaint 4 ---
Asset ID: FW000512
    Equipment Description: Maintenance Shop
    Line No: Maintenance Shop
    Work Order Type: Rotary Valve
    Failure Type: Process Failure
    Downtime:

In [1]:
print("INTERPRETATION SUMMARY:")
print("""
The retrieved eMaint records suggest recurring corrective maintenance activity associated with process-related failures.
This indicates that some assets and failure categories may be appearing more than once and may deserve closer review.

The retrieved coffee downtime records suggest repeated line-level downtime events, including cases with very low uptime,
limited production output, and relatively high downtime. This points to possible recurring interruptions in production activity.

Overall, the prototype suggests possible repeated downtime patterns linked to maintenance history and production-line events.
These results support further review of repeat failures, affected equipment, and higher-downtime operating periods.
""")

INTERPRETATION SUMMARY:

The retrieved eMaint records suggest recurring corrective maintenance activity associated with process-related failures.
This indicates that some assets and failure categories may be appearing more than once and may deserve closer review.

The retrieved coffee downtime records suggest repeated line-level downtime events, including cases with very low uptime,
limited production output, and relatively high downtime. This points to possible recurring interruptions in production activity.

Overall, the prototype suggests possible repeated downtime patterns linked to maintenance history and production-line events.
These results support further review of repeat failures, affected equipment, and higher-downtime operating periods.



In [31]:
monthly_downtime = (
    df1_trend.groupby("month")
    .agg(
        total_work_orders=("wo_no.", "count"),
        total_downtime_hours=("downtime", "sum"),
        avg_downtime_hours=("downtime", "mean")
    )
    .sort_index()
)
display(monthly_downtime)

,total_work_orders,total_downtime_hours,avg_downtime_hours
month,,,
2025-01,419,322.51,0.769714
2025-02,517,367.76,0.711335
2025-03,560,428.55,0.765268
2025-04,555,337.24,0.607640
2025-05,553,348.42,0.630054
2025-06,561,434.49,0.774492
2025-07,599,392.21,0.654775
2025-08,507,327.73,0.646410
2025-09,474,277.70,0.585865
